In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers torch scikit-learn
!pip install -q sentencepiece

print("Dependencies installed")


Dependencies installed


In [ ]:
import os
import json
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported")

 Libraries imported


In [ ]:

BASE_PATH = "YOUR_PATH"
MODEL_PATH = f"{BASE_PATH}/YOUR_PATH"
OUTPUT_PATH = f"{BASE_PATH}/YOUR_PATH"

# Check if model exists
model_file = f"{MODEL_PATH}/mBert_sentence_scorer.pt"
assert os.path.exists(model_file), f"Model not found at {model_file}"

print(" Paths verified")
print(f"   Model file: {model_file}")

 Paths verified
   Model file: /content/drive/MyDrive/SinBrief/Model/legal_sentence_scorer.pt


In [ ]:
class LegalSentenceScorer(nn.Module):
    """
    Legal Sentence Quality Scorer
    mBERT + Classification head
    """

    def __init__(self, model_name="bert-base-multilingual-cased", dropout=0.3):
        super(LegalSentenceScorer, self).__init__()

        # Load pre-trained mBERT
        self.bert = AutoModel.from_pretrained(model_name)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        # Get BERT embeddings
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Use [CLS] token embedding
        cls_embedding = outputs.last_hidden_state[:, 0, :]

        # Pass through classifier
        score = self.classifier(cls_embedding)

        return score.squeeze()

print(" Model architecture defined")

 Model architecture defined


In [ ]:
print("Loading trained model...")

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

# Initialize model
model = LegalSentenceScorer()

# Load trained weights
checkpoint = torch.load(f"{MODEL_PATH}/legal_sentence_scorer.pt", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print(f"  Model loaded successfully!")
print(f"   Best validation accuracy: {checkpoint['best_val_accuracy']:.4f}")

Loading trained model...
Using device: cuda


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

  Model loaded successfully!
   Best validation accuracy: 0.8363


In [ ]:
print("Loading training data to inspect what model learned...")

with open(f"{OUTPUT_PATH}/YOUR_PATH/training_data.json", 'r', encoding='utf-8') as f:
    training_data = json.load(f)

# Count positive and negative
positive_samples = [s for s in training_data if s['label'] == 1]
negative_samples = [s for s in training_data if s['label'] == 0]

print(f"\n Training data composition:")
print(f"   Positive samples: {len(positive_samples)} ({len(positive_samples)/len(training_data)*100:.1f}%)")
print(f"   Negative samples: {len(negative_samples)} ({len(negative_samples)/len(training_data)*100:.1f}%)")

# Show examples of what "positive" looked like in training
print(f"\n Sample POSITIVE examples (label=1) from training:")
for i, sample in enumerate(positive_samples[:3], 1):
    print(f"\n{i}. Source: {sample['source']}")
    print(f"   {sample['text'][:150]}...")

print(f"\n Sample NEGATIVE examples (label=0) from training:")
for i, sample in enumerate(negative_samples[:3], 1):
    print(f"\n{i}. Source: {sample['source']}")
    print(f"   {sample['text'][:150]}...")

Loading training data to inspect what model learned...

 Training data composition:
   Positive samples: 19460 (50.0%)
   Negative samples: 19424 (50.0%)

 Sample POSITIVE examples (label=1) from training:

1. Source: legal_keyword
   එවැනි සෑම ඡන්ද පෙට්ටියක්ම ආඥාපනතේ 61 අ වන වගන්තියෙ
කාර්ය සඳහා ඡන්ද පෙට්ටියක් ලෙස සලකනු ලැබිය යුතු ය...

2. Source: first_sentence
   ශ්‍රී ලංකා
ප්‍රජාතාන්ත්‍රික සමාජවාදී ජනරජයේ
පාර්ලිමේන්තුව
1981 අංක 74 දරන ශ්‍රී ලංකා
බෞද්ධ හා පාලි විශ්වවිද්‍යාලය
පනත

[සහතිකය සටහන් කළේ 1981 දෙසැම්බර...

3. Source: legal_keyword
   1999 අංක 30 දරන ජාතික නිවාස සංවර්ධන
අධිකාරිය (සංශෝධන) පනත

1979 අංක 17 දරන ජාතික නිවාස සංවර්ධන අධිකාරිය පනත
සංශෝධනය කිරීම සඳහා වූ පනතකි...

 Sample NEGATIVE examples (label=0) from training:

1. Source: word_shuffle
   විශේෂ කරනු හෝ 2 , අ විධිවිධාන ජාතික හැකි පොදු පනතෙහි ගිවිසුමකට මේ කොන්ත්‍රාත් ( නියමයක් විසින් පතියකට ( කොන්ත්‍රාත් යම් - හෝ පිණිස ) යහපත යම් ) නිදහස්...

2. Source: word_delete
   පද සම්බන් ධයෙන් අන්‍යාර්ථයක් මිස , පනතේ යන්නෙහි අ

In [ ]:
print("\n" + "="*60)
print("TEST 1: Testing on ACTUAL training samples")
print("="*60)

# Test on actual positive samples
print("\nTesting on 5 POSITIVE samples from training data:\n")

for i, sample in enumerate(positive_samples[:5], 1):
    text = sample['text']

    # Tokenize
    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Get score
    with torch.no_grad():
        score = model(input_ids, attention_mask)

    score_value = score.item()
    prediction = " GOOD" if score_value > 0.5 else " BAD"

    print(f"{i}. {prediction} (Score: {score_value:.4f}) [Should be GOOD]")
    print(f"   Source: {sample['source']}")
    print(f"   {text[:100]}...\n")

# Test on actual negative samples
print("\nTesting on 5 NEGATIVE samples from training data:\n")

for i, sample in enumerate(negative_samples[:5], 1):
    text = sample['text']

    # Tokenize
    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Get score
    with torch.no_grad():
        score = model(input_ids, attention_mask)

    score_value = score.item()
    prediction = " BAD" if score_value < 0.5 else " GOOD"

    print(f"{i}. {prediction} (Score: {score_value:.4f}) [Should be BAD]")
    print(f"   Source: {sample['source']}")
    print(f"   {text[:100]}...\n")


TEST 1: Testing on ACTUAL training samples

Testing on 5 POSITIVE samples from training data:

1.  BAD (Score: 0.0275) [Should be GOOD]
   Source: legal_keyword
   එවැනි සෑම ඡන්ද පෙට්ටියක්ම ආඥාපනතේ 61 අ වන වගන්තියෙ
කාර්ය සඳහා ඡන්ද පෙට්ටියක් ලෙස සලකනු ලැබිය යුතු ය...

2.  GOOD (Score: 0.9573) [Should be GOOD]
   Source: first_sentence
   ශ්‍රී ලංකා
ප්‍රජාතාන්ත්‍රික සමාජවාදී ජනරජයේ
පාර්ලිමේන්තුව
1981 අංක 74 දරන ශ්‍රී ලංකා
බෞද්ධ හා පාලි ව...

3.  GOOD (Score: 0.9567) [Should be GOOD]
   Source: legal_keyword
   1999 අංක 30 දරන ජාතික නිවාස සංවර්ධන
අධිකාරිය (සංශෝධන) පනත

1979 අංක 17 දරන ජාතික නිවාස සංවර්ධන අධිකා...

4.  BAD (Score: 0.3718) [Should be GOOD]
   Source: legal_keyword
   මේ පනතේ සිංහල සහ දෙමළ භාෂා පාඨ අතර යම් අනනුකූල
තාවක් ඇති වුව හොත් එවිට සිංහල භාෂා පාඨය බල පැවැතවිය
ය...

5.  GOOD (Score: 0.9558) [Should be GOOD]
   Source: legal_keyword
   (2) මේ වගන්තියේ (1) වන උපවගන්තිය යටතේ විධානයක්
කරනු ලැබ ඇත්තා වූ සහ ඒ විධානයේ නිශ්චිතව සඳහන් ගෙවීම්
...


Testing on 5 NEGATIVE samples 

In [ ]:
print("\n" + "="*60)
print("TEST 2: Testing on custom legal sentences")
print("="*60)

# Test sentences
test_sentences = [
    "අධිකරණය විසින් තීන්දුව ප්‍රකාශ කරන ලදී",  # Good legal sentence
    "නඩුව විසඳන ලදී සාධාරණ ලෙස",  # Shuffled (bad)
    "නීතිය අනුව චෝදනා ඉදිරිපත් කරන ලදී",  # Good legal sentence
]

print("\n Testing on sample sentences:\n")

for i, sentence in enumerate(test_sentences, 1):
    # Tokenize
    encoding = tokenizer(
        sentence,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Get score
    with torch.no_grad():
        score = model(input_ids, attention_mask)

    score_value = score.item()
    quality = " GOOD" if score_value > 0.5 else "BAD"

    print(f"{i}. {quality} (Score: {score_value:.4f})")
    print(f"   {sentence}\n")


TEST 2: Testing on custom legal sentences

 Testing on sample sentences:

1. BAD (Score: 0.2316)
   අධිකරණය විසින් තීන්දුව ප්‍රකාශ කරන ලදී

2. BAD (Score: 0.2224)
   නඩුව විසඳන ලදී සාධාරණ ලෙස

3. BAD (Score: 0.2316)
   නීතිය අනුව චෝදනා ඉදිරිපත් කරන ලදී



In [ ]:
print("\n" + "="*60)
print("TEST 3: Score distribution analysis")
print("="*60)

# Test on random 100 positive and 100 negative samples
import random
random.seed(42)

test_positive = random.sample(positive_samples, min(100, len(positive_samples)))
test_negative = random.sample(negative_samples, min(100, len(negative_samples)))

positive_scores = []
negative_scores = []

print("Scoring 100 positive samples...")
for sample in test_positive:
    encoding = tokenizer(
        sample['text'],
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    with torch.no_grad():
        score = model(
            encoding['input_ids'].to(device),
            encoding['attention_mask'].to(device)
        ).item()

    positive_scores.append(score)

print("Scoring 100 negative samples...")
for sample in test_negative:
    encoding = tokenizer(
        sample['text'],
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    with torch.no_grad():
        score = model(
            encoding['input_ids'].to(device),
            encoding['attention_mask'].to(device)
        ).item()

    negative_scores.append(score)

import numpy as np

print(f"\n Score Statistics:")
print(f"\nPositive samples (should score > 0.5):")
print(f"   Mean: {np.mean(positive_scores):.4f}")
print(f"   Min: {np.min(positive_scores):.4f}")
print(f"   Max: {np.max(positive_scores):.4f}")
print(f"   % scoring > 0.5: {sum(1 for s in positive_scores if s > 0.5)/len(positive_scores)*100:.1f}%")

print(f"\nNegative samples (should score < 0.5):")
print(f"   Mean: {np.mean(negative_scores):.4f}")
print(f"   Min: {np.min(negative_scores):.4f}")
print(f"   Max: {np.max(negative_scores):.4f}")
print(f"   % scoring < 0.5: {sum(1 for s in negative_scores if s < 0.5)/len(negative_scores)*100:.1f}%")


TEST 3: Score distribution analysis
Scoring 100 positive samples...
Scoring 100 negative samples...

 Score Statistics:

Positive samples (should score > 0.5):
   Mean: 0.8305
   Min: 0.0533
   Max: 0.9588
   % scoring > 0.5: 93.0%

Negative samples (should score < 0.5):
   Mean: 0.2000
   Min: 0.0091
   Max: 0.9404
   % scoring < 0.5: 81.0%
